# Features of Claude



In [1]:
%load_ext autoreload
%autoreload 2

# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

from my_chat_utils_with_tools import (
    add_user_message,
    add_assistant_message,
    chat,
    add_user_media_message,
    text_from_message,
)
from rich.console import Console

load_dotenv(override=True)

client = Anthropic()
MODEL = "claude-sonnet-4-5"
console = Console(force_jupyter=False)

## Extended Thinking Capability

Extended Thinking provides Claude with a private "scratchpad" to deliberate through complex, multi-step problems before generating a response. Its purpose is to boost accuracy and reasoning depth on difficult tasks where standard prompting falls short, while offering transparent visibility into Claude's thought process through verifiable, cryptographically secured reasoning blocks alongside the final answer.

In [11]:
# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [12]:
messages = []

add_user_message(messages, thinking_test_str)

chat(client, MODEL, messages, thinking=True)

Message(id='msg_011CewenMWiAGxbjn4A1HSLL', container=None, content=[RedactedThinkingBlock(data='EtsDCpsBCBEYAipAlFWsooD+1e/XQdHVUiKjRx7ZuCDxIxBkjd3bbMYWnahLLsB0I4OfJFjaAtgoyS6lCj5lHhPfyHXULMiWxrCJIjIaY2xhdWRlLXNvbm5ldC00LTUtMjAyNTA5Mjk4AEIIdGhpbmtpbmdaJDgwMDk5ZGJmLWFiNTQtNGE4Mi05ZmNlLTgwMmM0MTk0MTFjZKgB8OeP1QYSDIPyqEtt2MEnJkdnkBoMyoP9Wupdq8DJ0eY1IjACNOKjJT/I0GXkfmp5ebO2sYA9zJLn04SXus+2sAQZ6f9im2Qof0ECRR9EEI7JuEAq7AFDzNqAlO1p817F/bk7+uMWvfFXWsrIWkUbVek+l55G1scKUbihFoje5peJ5FpJCxG8p234CCDgGCMD9+asiWl5z9q9ezJCMqMYEYbPV06hCU2mLIowA/UlqhWt/GTYLggR+jDt3HjrUHON5Yv+w7gltU1yVIsA2ueHtKFBYonXyKQIcca1LsTYedDwQIlyENDbUmGnPGKdz7tLLHYOc+uHFZgRTBEe4TrmxVUfR+HZaIQ2SHs+OfM+v7LTaBU11IygQ1g67iWq4UtjNak+t5D929MfSagyXCd/I7ADfmshkIe09iE0H6OrCIMK1BgB', type='redacted_thinking'), TextBlock(citations=None, text='I don\'t have any special response to that string. It looks like you\'ve shared a text string that mentions "redacted thinking," but I don\'t have any hidden modes or special triggers.\n\nIs there somet

## Images Support

Claude’s Image Support equips the model with multimodal vision to examine, describe, compare, and extract insights from visual data alongside text. Its purpose is to automate complex visual inspection—such as object counting, property risk assessment, and scene analysis—by applying structured, step-by-step prompt engineering to images, enabling reliable decision-making and data extraction from visual inputs within existing conversational workflows.

In [6]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [7]:
# read in image data

with open("code_images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
params = {
    "messages": messages,
    "media_bytes": image_bytes,
    "user_message": prompt,
    "media_type": "image/png",
}
# add_user_media_message(messages, image_bytes, prompt, "image/png")
add_user_media_message(**params)
# print(f"Added media message: {messages[-1]}\n------\n")

response = chat(client, MODEL, messages)
print(response.content[0].text)

# Satellite Image Fire Risk Analysis

**1. Residence Identification:**
The primary residence is a single-story structure with a gray/light-colored roof located in the center of the image, featuring an irregular rectangular footprint with what appears to be attached garage sections, and is completely surrounded by dense tree canopy.

**2. Tree Overhang Analysis:**
Multiple mature trees have canopies that extend directly over approximately 50-75% of the residence's roof, with particularly dense overhang visible on the northern, eastern, and western portions of the structure, while only limited sections of the roof appear clear of tree coverage.

**3. Fire Risk Assessment:**
The overhanging trees create multiple ember catch points across the majority of the roof surface, establish direct fuel continuity between the surrounding forest canopy and the structure, and provide numerous pathways for fire to transfer from wildland vegetation directly onto the building.

**4. Defensible Space Iden

## PDF Support

Claude's PDF support enables direct reading and analysis of PDF files as a unified document processing solution. Its purpose is to extract and understand full-context document data beyond plain text—including embedded images, charts, formatting, and complex tables—using an API structure similar to image processing, allowing developers to perform deep summaries, structured data analysis, and multi-element extraction effortlessly.

PDF support uses almost the same code as image support, you'll open the PDF file, encode it as `utf-8` and then call `add_user_media_message` with `media_type = 'application/pdf'`. 

Here is the image:

![Earth.pdf](../lessons/images/earth_pdf.png)

And here is the code snippet, asking Claude to summarize the document.

In [8]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
doc_prompt = "Summarize the document in one sentence"

params = {
    "messages": messages,
    "media_bytes": file_bytes,
    "user_message": doc_prompt,
    "media_type": "application/pdf",
}
# add_user_media_message(messages, image_bytes, prompt, "image/png")
add_user_media_message(**params)
# print(f"Added media message: {messages[-1]}\n------\n")

response = chat(client, MODEL, messages)
print(response.content[0].text)

This Wikipedia article provides a comprehensive overview of Earth, describing it as the third planet from the Sun and the only known astronomical object to harbor life, detailing its physical characteristics, formation about 4.5 billion years ago, and its dynamic systems including oceans, atmosphere, and climate.


## Citations

The Citations feature in Claude grounds generated answers in user-provided documents by linking claims directly back to source material. Its purpose is to eliminate "black-box" responses and build trust, providing structured metadata—such as exact quoted text and page or character locations—so users can easily verify facts, trace information to authoritative sources, and explore surrounding context.

In [10]:
with open("article.txt", "r", encoding="utf-8") as f:
    article_text = f.read()

messages = []
citation_prompt = "How were Earth's atmosphere and oceans were formed?"

params = {
    "messages": messages,
    "media_bytes": article_text,
    "user_message": citation_prompt,
    "media_type": "text/plain",
    "title": "Earth Article",
    "enable_citations": True,
}
# add_user_media_message(messages, image_bytes, prompt, "image/png")
add_user_media_message(**params)
# console.print(messages[-1])
# print(f"Added media message: {messages[-1]}\n------\n")

response = chat(client, MODEL, messages)
console.print(response)

Message(
    id='msg_011CfVpQHNbKhtpRXihnNUGt',
    container=None,
    content=[
        TextBlock(
            citations=[
                CitationCharLocation(
                    cited_text="[42]\nAfter formation\nEarth's atmosphere and 
oceans were formed by volcanic activity and outgassing.\n",
                    document_index=0,
                    document_title='Earth Article',
                    end_char_index=9976,
                    file_id=None,
                    start_char_index=9876,
                    type='char_location'
                )
            ],
            text="Earth's atmosphere and oceans were formed by volcanic 
activity and outgassing.",
            type='text'
        ),
        TextBlock(citations=None, text=' ', type='text'),
        TextBlock(
            citations=[
                CitationCharLocation(
                    cited_text='[43] Water vapor from\nthese sources condensed 
into the oceans, augmented by water and ice from asteroids, pr

In [11]:
def display_response(response):
    """Display every content block in a chat() response.

    Handles the block types Claude can return in any response shape:
    plain text (image/PDF analysis), text with citations, extended-thinking
    (including redacted), and tool calls (client-side or server-side).
    Unrecognized block types are shown generically rather than dropped.
    """
    for block in response.content:
        block_type = getattr(block, "type", None)

        if block_type == "text":
            print(block.text)
            citations = getattr(block, "citations", None)
            if citations:
                for citation in citations:
                    title = getattr(citation, "document_title", None)
                    cited_text = getattr(citation, "cited_text", None)
                    loc_type = getattr(citation, "type", None)

                    print(f"  [citation] document: {title}")
                    print(f"             quote: {cited_text!r}")

                    if loc_type == "char_location":
                        print(
                            f"             location: chars "
                            f"{citation.start_char_index}-{citation.end_char_index}"
                        )
                    elif loc_type == "page_location":
                        print(
                            f"             location: pages "
                            f"{citation.start_page_number}-{citation.end_page_number}"
                        )
                    elif loc_type == "content_block_location":
                        print(
                            f"             location: content block "
                            f"{citation.start_block_index}-{citation.end_block_index}"
                        )

        elif block_type == "thinking":
            print(f"[thinking]\n{block.thinking}")

        elif block_type == "redacted_thinking":
            print("[redacted thinking block - content encrypted, not visible]")

        elif block_type in ("tool_use", "server_tool_use"):
            print(f"[tool call] {block.name}(input={block.input})")

        else:
            # e.g. web_search_tool_result, bash_code_execution_tool_result,
            # mcp_tool_result, or any future block type
            print(f"[{block_type} block]\n{block}")

        print()

    if response.stop_reason == "tool_use":
        print(
            "-- stop_reason='tool_use': Claude is waiting on tool_result(s) to continue --"
        )


display_response(response)

Earth's atmosphere and oceans were formed by volcanic activity and outgassing.
  [citation] document: Earth Article
             quote: "[42]\nAfter formation\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\n"
             location: chars 9876-9976

 

Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.
  [citation] document: Earth Article
             quote: '[43] Water vapor from\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\nand comets.\n'
             location: chars 9976-10108



## Prompt Caching in Action

In [2]:
from my_tool_schemas import (
    db_query_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    get_current_datetime_schema,
)

In [3]:
# load a gigantic 6k tokens prompt, which is meant to serve
# as a system prompt (& should be cached)
code_prompt: str = ""

with open("./prompt_with6ktokens.txt", "r", encoding="utf-8") as f:
    code_prompt = f.read()

# show first 100 characters
print(code_prompt[:100])

# Javascript Code Generator for Document Analysis Flow

You are an expert Javascript code generator.


In [4]:
# let's first see the output with no system prompt, no tools, no caching
# NOTE: our system prompt & tools are now cached in the chat call

messages = []

add_user_message(messages, "what is 1+1")

response = chat(client, MODEL, messages)
console.print(response)

# Observe the usage fields of the response

Message(
    id='msg_011CfVt3YYXYrnBRvCjKCvCY',
    container=None,
    content=[TextBlock(citations=None, text='1+1 = 2', type='text')],
    model='claude-sonnet-4-5-20250929',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(
            ephemeral_1h_input_tokens=0,
            ephemeral_5m_input_tokens=0
        ),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=0,
        inference_geo='not_available',
        input_tokens=13,
        output_tokens=11,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    ),
    diagnostics=None
)


In [ ]:
# now let's add tools (which are cached) & see the response field
tools = [
    db_query_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    get_current_datetime_schema,
]

messages = []

add_user_message(messages, "what is 1+1")

response = chat(client, MODEL, messages, tools=tools)
console.print(response)

# Again, observe the usage fields of the response
# now you see Claude adding cache_creation_input_tokens, meaning that
# it has written ~1.7k tokens into the cache

Message(
    id='msg_011CfVt629G4A5udcB7mDpF7',
    container=None,
    content=[
        TextBlock(
            citations=None,
            text='1 + 1 = 2\n\nThis is a basic arithmetic operation where 
adding 1 and 1 together equals 2.',
            type='text'
        )
    ],
    model='claude-sonnet-4-5-20250929',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(
            ephemeral_1h_input_tokens=0,
            ephemeral_5m_input_tokens=1774
        ),
        cache_creation_input_tokens=1774,
        cache_read_input_tokens=0,
        inference_geo='not_available',
        input_tokens=333,
        output_tokens=34,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    ),
    diagnostics=None
)


In [ ]:
# now make a follow up request, without changing anything
# observe the usage field
messages = []

add_user_message(messages, "what is 1+1")

response = chat(client, MODEL, messages, tools=tools)
console.print(response)

# observe the usage fields
# now you have a cache_read_input_tokens=1774, which means
# claude read the tools schemas (~1.7k) from cache!
# these 2 calls in sequence means that we are able to save our tools-schema
# into the cache & retrieve them from cache in future

Message(
    id='msg_011CfVtJTWLkATT69kLHZ6cx',
    container=None,
    content=[TextBlock(citations=None, text='1 + 1 = 2', type='text')],
    model='claude-sonnet-4-5-20250929',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(
            ephemeral_1h_input_tokens=0,
            ephemeral_5m_input_tokens=0
        ),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=1774,
        inference_geo='not_available',
        input_tokens=333,
        output_tokens=13,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    ),
    diagnostics=None
)


In [ ]:
# even if we change our user message, we'll still read the
# tools info from cache - let's confirm
# NOTE: our user message is different from before

messages = []

add_user_message(messages, "what is 5-7?")

response = chat(client, MODEL, messages, tools=tools)
console.print(response)

# observe the usage fields
# you still have cache_read_input_tokens=1774, which means
# claude read the tools schemas (~1.7k) from cache!

Message(
    id='msg_011CfVtXXPs8TMVqNpbaQHGC',
    container=None,
    content=[
        TextBlock(
            citations=None,
            text='The answer to 5 - 7 is **-2**.',
            type='text'
        )
    ],
    model='claude-sonnet-4-5-20250929',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(
            ephemeral_1h_input_tokens=0,
            ephemeral_5m_input_tokens=0
        ),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=1774,
        inference_geo='not_available',
        input_tokens=334,
        output_tokens=18,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    ),
    diagnostics=None
)


In [ ]:
# adding system prompt is very simple

# notice that the first time we add system prompt, ther'll be a cache write
messages = []

add_user_message(messages, "what is 5-7?")

response = chat(client, MODEL, messages, tools=tools, system_prompt=code_prompt)
console.print(response)

# now I have a cache_creation_input_tokens=6331 (the new ~6k system prompt is added
# to the cache), and cache_read_input_tokens=1774 (the ~1.7k tools prompt are read
# from cache!)

Message(
    id='msg_011CfVtsq9zvKa29deoNHeCp',
    container=None,
    content=[TextBlock(citations=None, text='5 - 7 = -2', type='text')],
    model='claude-sonnet-4-5-20250929',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(
            ephemeral_1h_input_tokens=0,
            ephemeral_5m_input_tokens=6331
        ),
        cache_creation_input_tokens=6331,
        cache_read_input_tokens=1774,
        inference_geo='not_available',
        input_tokens=330,
        output_tokens=13,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    ),
    diagnostics=None
)


In [ ]:
# run the same cell again, with different user message
# I'd expect claude to read BOTH system prompt & tools from cache

# notice that the first time we add system prompt, ther'll be a cache write
messages = []

add_user_message(messages, "what is 12 * 5?")

response = chat(client, MODEL, messages, tools=tools, system_prompt=code_prompt)
console.print(response)

# now we have cache_creation_input_tokens=0 (no new tokens added to cache), and
# cache_read_input_tokens=8105 (= 6331 + 1774), which is exactly the sum of system
# prompt token + tools tokens

Message(
    id='msg_011CfVu3FhtCeoH4a4HQFPUJ',
    container=None,
    content=[TextBlock(citations=None, text='12 * 5 = 60', type='text')],
    model='claude-sonnet-4-5-20250929',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(
            ephemeral_1h_input_tokens=0,
            ephemeral_5m_input_tokens=0
        ),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=8105,
        inference_geo='not_available',
        input_tokens=332,
        output_tokens=13,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    ),
    diagnostics=None
)


**NOTE**: If we were to change the tool definitions or system prompt in any way between the successful calls, then the cache would be invalidated. Meaning, Claude would create a new cache write for the modified tools/system-prompt in the first call and subsequent read from cache, if no changes to tools/system prompt!

## Files API and Code Execution

The Anthropic API offers two powerful features that work exceptionally well together: the Files API and Code Execution. While they might seem separate at first, combining them opens up some really interesting possibilities for delegating complex tasks to Claude.

* **The Files API** provides an alternative way to handle file uploads. Instead of encoding images or PDFs directly in your messages as base64 data, you can upload files ahead of time and reference them later.
* **The Code Execution API** is a server-based tool that doesn't require you to provide an implementation. You simply include a predefined tool schema in your request, and Claude can optionally execute Python code in an isolated Docker container.

The real power comes from using these features together. Since the Docker containers have no network access, the Files API becomes the primary way to get data in and out of the execution environment.

For example, you could upload a CSV file and ask Claude to do a comprehensive analysis, as we'll do here.

In [12]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

from my_chat_utils_with_tools import (
    add_user_message,
    add_assistant_message,
    chat,
    add_user_media_message,
    text_from_message,
    # we have added a new helper 'upload' to handle
    # file uploads
    upload,
)
from rich.console import Console

load_dotenv(override=True)

# notice the change in the client definition

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
MODEL = "claude-sonnet-4-5"
console = Console(force_jupyter=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [15]:
# upload our data file
file_metadata = upload(client, "streaming.csv")
file_metadata

BetaFileMetadata(id='file_016FGzbEhvXDoaFUphCNfarS', created_at=datetime.datetime(2026, 9, 28, 13, 42, 14, 953258, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

In [ ]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

# let's ask Claude to analyze the uploaded file
# WARNING: this will take some time to run on server!
tools_list = [
    {"type": "code_execution_20250825", "name": "code_execution"},
]
response = chat(client, MODEL, messages, tools=tools_list)
console.print(response)

Message(
    id='msg_011CfVwSaBk8X7vyhS1nuepY',
    container=Container(
        id='container_0147nJ9XSkyKoMoRT7A278DY',
        expires_at=datetime.datetime(2026, 9, 28, 14, 47, 22, 67109, 
tzinfo=TzInfo(0)),
        skills=None
    ),
    content=[
        TextBlock(
            citations=None,
            text="I'll help you analyze the drivers of churn in the streaming 
dataset. Let me start by exploring the data and then perform a detailed 
analysis.",
            type='text'
        ),
        ServerToolUseBlock(
            id='srvtoolu_011YCSkec4AuUenMXHhDKRPN',
            caller=None,
            input={'command': 'cd $INPUT_DIR && head -20 streaming.csv'},
            name='bash_code_execution',
            type='server_tool_use'
        ),
        BashCodeExecutionToolResultBlock(
            content=BashCodeExecutionResultBlock(
                content=[],
                return_code=0,
                stderr='',
                stdout='UserID,SubscriptionTier,TotalViewin

In [18]:
# now let's see the plot
download_file(msg_011CfVwSaBk8X7vyhS1nuepY)

NameError: name 'msg_011CfVwSaBk8X7vyhS1nuepY' is not defined